In [4]:
import xarray as xr
import pandas as pd
from pathlib import Path

# --- bbox (same as kelp) ---
lat_min, lat_max = 33.8, 34.8
lon_min, lon_max = -120.8, -119.0

# --- find files ---
oisst_dir = Path("../1_DATA/raw")  # adjust if your OISST files are in raw/oisst/
files = sorted(oisst_dir.rglob("oisst_socal_*.nc"))
if not files:
    raise FileNotFoundError(f"No files matched under {oisst_dir.resolve()}")

print("Found", len(files), "files")
print("First:", files[0].name, "| Last:", files[-1].name)

out_csv = Path("../1_DATA/processed/oisst_socal_bbox_monthly.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

# --- open all files ---
ds = xr.open_mfdataset(files, combine="by_coords")

# --- detect coord/var names ---
def pick_first(existing, candidates):
    for c in candidates:
        if c in existing:
            return c
    return None

lat_name = pick_first(ds.coords, ["lat", "latitude"])
lon_name = pick_first(ds.coords, ["lon", "longitude"])
time_name = pick_first(ds.coords, ["time"])

sst_var = pick_first(ds.data_vars, ["sst", "SST", "analysed_sst", "sea_surface_temperature"])
if None in (lat_name, lon_name, time_name, sst_var):
    raise ValueError(f"Could not detect names. coords={list(ds.coords)}, data_vars={list(ds.data_vars)}")

# --- normalize lon to [-180, 180] if needed ---
if float(ds[lon_name].max()) > 180:
    ds = ds.assign_coords({lon_name: (((ds[lon_name] + 180) % 360) - 180)}).sortby(lon_name)

sst = ds[sst_var]

# --- subset bbox (handle descending lat) ---
sst_bbox = sst.sel({lat_name: slice(lat_min, lat_max), lon_name: slice(lon_min, lon_max)})
if sst_bbox.sizes.get(lat_name, 0) == 0:
    sst_bbox = sst.sel({lat_name: slice(lat_max, lat_min), lon_name: slice(lon_min, lon_max)})

# --- bbox mean (should now be 1D time) ---
sst_mean = sst_bbox.mean(dim=[lat_name, lon_name], skipna=True).squeeze()

# --- to pandas series ---
sst_daily = sst_mean.to_pandas()
sst_daily.index = pd.to_datetime(sst_daily.index)
sst_daily = sst_daily.sort_index()
sst_daily.name = "sst"

# --- monthly mean ---
sst_m = sst_daily.resample("MS").mean()  # month start
sst_m.name = "sst"

# --- anomaly: subtract month-of-year climatology ---
sst_anom_m = sst_m - sst_m.groupby(sst_m.index.month).transform("mean")

df_out = pd.DataFrame({"sst": sst_m, "sst_anom": sst_anom_m})
df_out.to_csv(out_csv, index=True)

print("Saved:", out_csv.resolve())
print("Range:", df_out.index.min(), "to", df_out.index.max(), "rows:", len(df_out))
print(df_out.head())

Found 13 files
First: oisst_socal_1984-01-01_1988-12-31.nc | Last: oisst_socal_test_2020.nc
Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_socal_bbox_monthly.csv
Range: 1984-01-01 00:00:00 to 2025-12-01 00:00:00 rows: 504
                  sst  sst_anom
time                           
1984-01-01  14.946989  0.861664
1984-02-01  14.871823  0.977874
1984-03-01  14.898540  1.227298
1984-04-01  13.516842  0.073137
1984-05-01  14.628971  0.585222


In [7]:
import pandas as pd
from pathlib import Path

sst_m_path = Path("../1_DATA/processed/oisst_socalv1_bbox_monthly.csv")
kelp_path  = Path("../1_DATA/processed/kelp_timeseries_socalv1_bbox_quarterly.csv")
out_path   = Path("../1_DATA/processed/oisst_features_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

# load
sst_m = pd.read_csv(sst_m_path, parse_dates=["time"])
sst_m = sst_m.set_index("time").sort_index()

df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

# quarterly features from monthly
q = sst_m.resample("QS")  # quarter start buckets

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})

# lag (1 quarter)
feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

# align to kelp quarters (using q-start)
feat = feat.reindex(kelp_qstart)
feat.index = kelp_times  # set index to kelp timestamps for easy join later

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_features_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_q_mean_lag1
1984-02-15   14.905784        1.022278       1.227298                  NaN
1984-05-15   14.784969        0.499183       0.839190             1.022278
1984-08-15   19.527659        2.072631       2.553732             0.499183
1984-11-15   15.633710       -0.445956      -0.300495             2.072631
1985-02-15   13.353732       -0.529773      -0.279811            -0.445956
1985-05-15   13.924952       -0.360834      -0.229975            -0.529773
1985-08-15   17.502159        0.047131       0.271622            -0.360834
1985-11-15   15.583540       -0.496127      -0.152170             0.047131
1986-02-15   14.299319        0.415813       0.623964            -0.496127
1986-05-15   14.245347       -0.040439       0.557709             0.415813
